# Financial Sentiment Analysis — QLoRA + Pinecone RAG
**Model**: LLaMA-2-7B | **Technique**: QLoRA | **Dataset**: Financial Phrasebank

Run cells in order from top to bottom.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/financial-llm/saved_model', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/cache', exist_ok=True)

print('Drive mounted and folders created.')

## Cell 2 — Set API Keys (from Colab Secrets)

In [ ]:
# Add your keys in Colab: left sidebar → 🔑 Secrets
# Keys needed: PINECONE_API_KEY, HF_TOKEN

from google.colab import userdata
import os

os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
os.environ['HF_TOKEN']         = userdata.get('HF_TOKEN')

# Cache HuggingFace datasets to Drive so we don't re-download each session
os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/financial-llm/cache'

print('API keys loaded.')

## Cell 3 — Clone Repo + Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/financial-llm-sentiment.git'
REPO_DIR = '/content/financial-llm-sentiment'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if repo already cloned
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print('Setup complete.')

## Cell 4 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu_name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    print(f'BF16 : {torch.cuda.is_bf16_supported()}')
else:
    print('No GPU detected. Go to Runtime → Change runtime type → GPU (A100).')

## Cell 5 — Prepare Data (preview)

In [ ]:
import sys
sys.path.insert(0, '/content/financial-llm-sentiment')

from data.prepare_data import load_raw_data, split_data

raw_data     = load_raw_data()
dataset_dict = split_data(raw_data)

print('\nSample from train set:')
print(dataset_dict['train'][0])

## Cell 6 — Train Model

In [ ]:
from model.train import train

# Runs full pipeline:
#   prepare_dataset → load_base_model → apply_lora → train → evaluate → save
trainer, tokenizer, rag_sentences = train()

## Cell 7 — Index Financial Sentences into Pinecone

In [ ]:
from rag.pinecone_utils import initialize_rag

# rag_sentences came from train() — reuse without reloading data
index, embed_model = initialize_rag(rag_sentences)

print(f'Pinecone index stats: {index.describe_index_stats()}')

## Cell 8 — Test Inference

In [ ]:
from model.inference import predict_sentiment, predict_with_rag

test_sentences = [
    'The company reported record profits this quarter.',
    'Markets remained flat amid economic uncertainty.',
    'Stock prices collapsed after the earnings miss.',
]

print('=== Without RAG ===')
for s in test_sentences:
    result = predict_sentiment(s, trainer.model, tokenizer)
    print(f'  [{result["label"].upper():8s}] {s}')

print('\n=== With RAG ===')
for s in test_sentences:
    result = predict_with_rag(s, trainer.model, tokenizer, index, embed_model)
    print(f'  [{result["label"].upper():8s}] {s}')
    print(f'  Context: {result["retrieved_context"][0][:80]}...')

## Cell 9 — Evaluate on Test Set

In [ ]:
from data.prepare_data import prepare_dataset
from model.evaluate import evaluate_on_testset

tokenized_dataset, _, _ = prepare_dataset()
results = evaluate_on_testset(trainer.model, tokenizer, tokenized_dataset['test'])

## Cell 10 — Attention Heatmap

In [ ]:
from model.evaluate import plot_attention_heatmap

plot_attention_heatmap(
    sentence  = 'The company reported record profits this quarter.',
    model     = trainer.model,
    tokenizer = tokenizer,
)

## Cell 11 — Benchmark Table

In [ ]:
from model.evaluate import build_benchmark_table

# Fill in the numbers after running each model
benchmark_results = {
    'BERT-base':   {'params_M': 110,  'time_min': 15, 'memory_gb': 2.1, 'f1': 0.0, 'accuracy': 0.0},
    'LLaMA-QLoRA': {'params_M': 7000, 'time_min': 45, 'memory_gb': 5.2, 'f1': results['f1_macro'], 'accuracy': results['accuracy']},
}

df = build_benchmark_table(benchmark_results)

## Cell 12 — Launch Gradio App

In [ ]:
# Run the full app with model + RAG already loaded
from model.inference import load_trained_model
from app.gradio_app import build_app
import app.gradio_app as app_module

# Inject already-loaded objects so startup() is not called again
app_module.model       = trainer.model
app_module.tokenizer   = tokenizer
app_module.index       = index
app_module.embed_model = embed_model

gradio_app = build_app()
gradio_app.launch(share=True)   # generates a public link valid for 72 hours

## Cell 13 — Push to GitHub (optional)

In [ ]:
# Run after making changes to any .py file
!git config user.email 'your@email.com'
!git config user.name  'Your Name'
!git add .
!git status
!git commit -m 'feat: complete training pipeline'
!git push origin main
# Enter your GitHub Personal Access Token when prompted for password